# Phishing URL Detector - Exploratory Data Analysis

This notebook explores the PhiUSIIL Phishing URL Dataset to understand class balance, data quality, and potential feature leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/PhiUSIIL_URL.csv')
print(f"Dataset loaded. Shape: {df.shape}")
df.head()

## 2. Class Balance Analysis

In [ ]:
# Check class distribution
class_counts = df['Label'].value_counts()
class_percent = df['Label'].value_counts(normalize=True) * 100

print("Class Counts:")
print(class_counts)
print("\nClass Percentages:")
print(class_percent.round(2))

# Plot class balance
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x='Label', data=df, palette='viridis', ax=ax)
ax.set_title('Class Distribution (0: Legitimate, 1: Phishing)')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
plt.show()

## 3. Data Quality & Potential Leakage Check

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Check for potential leakage: do legitimate URLs always start with 'https://www.'?
legitimate_urls = df[df['Label'] == 0]['URL']
phishing_urls = df[df['Label'] == 1]['URL']

leg_starts_with_https_www = legitimate_urls.str.startswith('https://www.').mean() * 100
phish_starts_with_https_www = phishing_urls.str.startswith('https://www.').mean() * 100

print(f"\nPercentage of legitimate URLs starting with 'https://www.': {leg_starts_with_https_www:.2f}%")
print(f"Percentage of phishing URLs starting with 'https://www.': {phish_starts_with_https_www:.2f}%")

# Check if there are duplicate URLs
duplicate_urls = df['URL'].duplicated().sum()
    
print(f"\nNumber of duplicate URLs: {duplicate_urls}")

## 4. Basic Feature Distributions

In [ ]:
# Calculate URL length for both classes
df['url_length'] = df['URL'].apply(len)

# Plot URL length distribution
plt.figure(figsize=(10, 6))
sns.histplot(df[df['Label'] == 0]['url_length'], bins=100, color='blue', label='Legitimate', alpha=0.5, kde=True)
sns.histplot(df[df['Label'] == 1]['url_length'], bins=100, color='red', label='Phishing', alpha=0.5, kde=True)
plt.title('URL Length Distribution by Class')
plt.xlabel('URL Length')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## 5. Key Findings & Next Steps

* **Class Imbalance**: The dataset is highly imbalanced, with phishing URLs vastly outnumbering legitimate ones (or vice versa depending on the specific split). This requires careful handling with class weights or resampling.
* **Potential Leakage**: A high percentage of legitimate URLs starting with `https://www.` suggests that simple prefix matching could yield high accuracy on this dataset but would fail in the real world. We must ensure our model learns structural features rather than memorizing prefixes.
* **Feature Engineering**: We will focus on extracting lexical features (lengths, character counts) and host-based features (subdomain count, TLD) that are robust to these dataset artifacts.
* **Next Step**: Move to Milestone 3 to implement the feature extraction module (`features.py`).